---
numbering: false
---

# 2.5: Linear equations in ℝ³


In [1]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys

_notes_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'myst.yml').exists())
if str(_notes_root) not in sys.path:
    sys.path.insert(0, str(_notes_root))
from plot_style import style_plotly

BLUE, ORANGE, PINK = '#3d81f6', 'orange', '#d81a60'


def base3():
    fig=style_plotly(go.Figure(),renderer='plotly_mimetype')
    axis=dict(range=[-7,10],dtick=2,tickfont=dict(size=11),showbackground=True,showspikes=False,
              backgroundcolor='white',gridcolor='#e5e7eb',zerolinecolor='#9ca3af')
    fig.update_layout(autosize=True,height=520,showlegend=False,font=dict(size=16),
                      margin=dict(l=0,r=0,t=15,b=0),
                      scene=dict(bgcolor='white',xaxis=dict(title=dict(text='x',font=dict(size=13)),**axis),yaxis=dict(title=dict(text='y',font=dict(size=13)),**axis),
                                 zaxis=dict(title=dict(text='z',font=dict(size=13)),**axis),aspectmode='cube',
                                 camera=dict(eye=dict(x=1.6,y=-2.1,z=1.3))))
    for direction in np.eye(3):
        line3(fig,-6*direction,8*direction,'#9ca3af',width=2)
    return fig


def line3(fig,start,end,color=BLUE,width=5,dash='solid'):
    fig.add_trace(go.Scatter3d(x=[start[0],end[0]],y=[start[1],end[1]],z=[start[2],end[2]],
                              mode='lines',line=dict(color=color,width=width,dash=dash),
                              hoverinfo='skip',showlegend=False))


def vec3(fig,end,label,color=BLUE,start=(0,0,0),offset=(0.25,0.25,0.35)):
    start,end=np.asarray(start,float),np.asarray(end,float)
    direction=(end-start)/np.linalg.norm(end-start)
    line3(fig,start,end-0.15*direction,color,width=7)
    fig.add_trace(go.Cone(x=[end[0]],y=[end[1]],z=[end[2]],u=[direction[0]],v=[direction[1]],w=[direction[2]],
                         anchor='tip',sizemode='absolute',sizeref=0.45,colorscale=[[0,color],[1,color]],
                         showscale=False,hoverinfo='skip'))
    pos=end+offset
    fig.add_trace(go.Scatter3d(x=[pos[0]],y=[pos[1]],z=[pos[2]],mode='text',text=[label],
                              textfont=dict(family='Palatino',color=color,size=20),hoverinfo='skip'))


def plane3(fig,normal,color=BLUE,opacity=0.24,d=0,bounds=(-5,7)):
    # Draw a complete rectangle in the plane, with room around every edge.
    n=np.asarray(normal,float)
    unit=n/np.linalg.norm(n)
    reference=np.eye(3)[np.argmin(np.abs(unit))]
    u=np.cross(unit,reference)
    u=u/np.linalg.norm(u)
    v=np.cross(unit,u)
    center=d*n/np.dot(n,n)
    lo,hi=bounds
    corners=np.array([center+a*u+b*v for a,b in
                      [(lo,lo),(hi,lo),(hi,hi),(lo,hi)]])
    fig.add_trace(go.Mesh3d(x=corners[:,0].tolist(),y=corners[:,1].tolist(),z=corners[:,2].tolist(),
                            i=[0,0],j=[1,2],k=[2,3],
                            color=color,opacity=opacity,flatshading=True,
                            lighting=dict(ambient=1,diffuse=0,specular=0,fresnel=0,roughness=1),
                            showscale=False,hoverinfo='skip'))
    for i in range(4):
        line3(fig,corners[i],corners[(i+1)%4],color,width=2)
    lower=min(fig.layout.scene.xaxis.range[0],float(corners.min())-1)
    upper=max(fig.layout.scene.xaxis.range[1],float(corners.max())+1)
    fig.update_scenes(xaxis_range=[lower,upper],yaxis_range=[lower,upper],
                      zaxis_range=[lower,upper])



In [Chapter 2.4](02-04.ipynb), we described lines and planes through the origin using spans and parametric equations. Here, we'll describe them using **linear equations**:

$$ax + by + cz = d$$

Recall, a **homogenous** linear equation is one whose right-hand side above is $0$. We will focus on homogenous equations here, and explore nonzero values of $d$ in [Chapter 2.6](./02-06.ipynb).

---

## Why does one linear equation describe a plane?

In $\mathbb R^2$, a linear equation of the form $ax+by=c$, with $a,b$ not both zero, describes a line.

**In $\mathbb R^3$, a single linear equation with at least one nonzero coefficient describes a plane.** For now, we'll focus on homogeneous equations:

$$ax+by+cz=0.$$

Why a plane instead of a line? For example, consider

$$x-2y+z=0,$$

or equivalently,

$$z=-x+2y.$$

We can plug in **any** $x$ and **any** $y$ to get an output $z$. For instance, when $x=1$ and $y=1$, we get $z=1$. Allowing every possible pair of $x$ and $y$ gives us a plane.



In [2]:
fig=base3()
plane3(fig,(1,-2,1))
fig.show()


```{figure} #plot-25-first-plane
:label: fig-plot-25-first-plane
:class: course-caption
:alt: The plane x minus 2y plus z equals zero.

The plane $x-2y+z=0$, or $z=-x+2y$. Every pair of $x$ and $y$ determines a point on the plane.
```




But a line only works for very specific pairs of $x$ and $y$. Recall the line spanned by $\vec v=\begin{bmatrix}3\\4\\5\end{bmatrix}$ from [Chapter 2.4](02-04.ipynb). There's no point on this line that has $x=3$ and $y=1$. Rather, when $x=3$, $y$ is forced to be $4$, and $z$ is forced to be $5$.

So we cannot describe this line by a formula for $z$ in terms of $x$ and $y$ that lets us plug in **any** $x$ and **any** $y$: most pairs of $x$ and $y$-values do not lie on the line.



In [3]:
fig=base3()
plane3(fig,(0,1,0))
fig.show()


```{figure} #plot-25-xz-plane
:label: fig-plot-25-xz-plane
:class: course-caption
:alt: The xz-plane, where y equals zero.

For example, $y=0$ gives us the $xz$-plane.
```


In [4]:
fig=base3()
plane3(fig,(2,-3,9))
fig.show()


```{figure} #plot-25-other-plane
:label: fig-plot-25-other-plane
:class: course-caption
:alt: The plane 2x minus 3y plus 9z equals zero.

The equation $2x-3y+9z=0$ describes another plane through the origin.
```


---

## Linear equations for planes

Let's return to the plane from [Chapter 2.4](02-04.ipynb):

$$P=\operatorname{span}(\vec v_1,\vec v_2),\qquad
\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad
\vec v_2=\begin{bmatrix}5\\2\\-1\end{bmatrix}.$$

Note that we **don't** yet have a linear equation describing this plane: we've expressed the plane as the span of two vectors. How might we find a linear equation that describes this plane?


In [5]:
fig=base3()
plane3(fig,(1,-2,1))
vec3(fig,(3,4,5),'<i>v</i>⃗<sub>1</sub>',BLUE)
vec3(fig,(5,2,-1),'<i>v</i>⃗<sub>2</sub>',ORANGE,offset=(-0.8,0.2,0.4))
fig.show()




```{figure} #plot-25-recap
:label: fig-plot-25-recap
:class: course-caption
:alt: The blue plane contains the blue and orange spanning vectors.

The plane $P$ spanned by $\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix}$ and $\vec v_2=\begin{bmatrix}5\\2\\-1\end{bmatrix}$.
```



For now, we'll just tell it to you: an equation for this plane is

$$x-2y+z=0.$$

You should verify yourself that both $\vec v_1 = \begin{bmatrix} 3 \\ 4 \\ 5 \end{bmatrix}$ and $\vec v_2 = \begin{bmatrix} 5 \\ 2 \\ -1 \end{bmatrix}$ satisfy the equation above.

How could we have found this equation without guessing? The key is to find a vector perpendicular to the plane. Note that the equation above can be written as

$$\begin{bmatrix} x \\ y \\ z \end{bmatrix} \cdot \begin{bmatrix} 1 \\ -2 \\ 1 \end{bmatrix}.$$

All vectors that satisfy $x - 2y + z = 0$ are perpendicular to $\begin{bmatrix} 1 \\ -2 \\ 1 \end{bmatrix}$!

---

## Normal vectors

Remember that in [Chapter 2.1](02-01.ipynb), we saw that in the equation of a line in $\mathbb{R}^2$,

$$ax + by = c,$$

the vector $\begin{bmatrix} a \\ b\end{bmatrix}$ – found by reading the coefficients of $x$ and $y$ above – is orthogonal to the line above.

Let's try the same thing here. The coefficients on $x$, $y$, and $z$ in $$x-2y+z=0$$ imply the normal vector

$$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}.$$

In [6]:
fig=base3()
plane3(fig,(1,-2,1))
n=np.array([1,-2,1])
line3(fig,-2*n,2*n,PINK,width=3)
vec3(fig,(3,4,5),'<i>v</i>⃗<sub>1</sub>',BLUE)
vec3(fig,(5,2,-1),'<i>v</i>⃗<sub>2</sub>',ORANGE,offset=(-0.6,0.2,0.4))
vec3(fig,n,'<i>w</i>⃗',PINK,offset=(0.5,-0.5,0.4))
fig.update_layout(scene_camera=dict(eye=dict(x=2,y=-1.1,z=1.1)))
fig.show()




```{figure} #plot-25-normal
:label: fig-24-normal
:class: course-caption
:alt: The pink normal vector w is perpendicular to the blue plane P.

The vector
$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}$ is perpendicular to the plane
$P$.
```




To show that $\vec w$ is perpendicular to $P$, we need to show that it's orthogonal to **every vector in $P$**. There are infinitely many such vectors, but we only need two dot products to get started:

$$\vec v_1\cdot\vec w=3(1)+4(-2)+5(1)=0,$$

$$\vec v_2\cdot\vec w=5(1)+2(-2)+(-1)(1)=0.$$

Remember that every vector in $P$ is a linear combination of $\vec v_1$ and $\vec v_2$. Since both dot products above are $0$, the dot product of $\vec w$ with any of their linear combinations is also $0$:

$$\begin{aligned}
(a\vec v_1+b\vec v_2)\cdot\vec w
&=a(\vec v_1\cdot\vec w)+b(\vec v_2\cdot\vec w)\\
&=a(0)+b(0)=0.
\end{aligned}$$

That's why checking the two spanning vectors is enough. It tells us that $\vec w$ is perpendicular to the entire plane.

:::{note} Definition: Normal vector to a plane
A **normal vector** to a plane is a nonzero vector perpendicular to every direction in the plane.

For a plane with equation $ax+by+cz=d$, the vector $\begin{bmatrix}a\\b\\c\end{bmatrix}$ is a normal vector, as is every scalar multiple of $\begin{bmatrix} a \\ b \\ c\end{bmatrix}$.
:::

For our plane through the origin (where $d=0$), we can write

$$P=\left\{\begin{bmatrix}x\\y\\z\end{bmatrix}:
\vec w\cdot\begin{bmatrix}x\\y\\z\end{bmatrix}=0\right\}.$$

The above form is sometimes called the **dot-product form** equation of a plane.

We'll write $P^\perp$ for the set of vectors orthogonal to every vector in $P$. Here it is the line

$$P^\perp=\operatorname{span}(\vec w).$$

This extends the perpendicular notation from [Chapter 2.1](02-01.ipynb) through [Chapter 2.3](02-03.ipynb).

- In $\mathbb R^2$, the vectors perpendicular to a line through the origin form another line.
- In $\mathbb R^3$, the vectors perpendicular to a plane through the origin form a line, while the vectors perpendicular to a line through the origin form a plane.

A normal vector is not unique. Using $3\vec w=\begin{bmatrix}3\\-6\\3\end{bmatrix}$ gives $3x-6y+3z=0$, which is the original equation multiplied by $3$. The plane does not change.



---

## Finding a normal vector for a plane

So far, we've checked a normal vector after being given an equation. What if we only know the spanning vectors? Let's find a normal to the same plane

$$P=\operatorname{span}(\vec v_1,\vec v_2),\qquad
\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad
\vec v_2=\begin{bmatrix}5\\2\\-1\end{bmatrix}.$$

Write the unknown normal as $\vec n=\begin{bmatrix}a\\b\\c\end{bmatrix}$. It must be orthogonal to both spanning vectors, so

$$3a+4b+5c=0,\qquad 5a+2b-c=0.$$

The second equation gives $c=5a+2b$. Substituting into the first gives

$$3a+4b+5(5a+2b)=0\quad\Longrightarrow\quad28a+14b=0.$$

Thus $b=-2a$ and $c=5a+2(-2a)=a$. Choosing $a=1$ gives

$$\vec n=\begin{bmatrix}1\\-2\\1\end{bmatrix}.$$

These are exactly the coefficients in the plane's equation:

$$x-2y+z=0.$$

Any nonzero choice of $a$ gives a scalar multiple of this normal and an equivalent equation for the same plane.

::::{tip} Activity 1
Find a nonzero normal to the plane spanned by

$$\vec u=\begin{bmatrix}5\\-7\\3\end{bmatrix},\qquad
\vec v=\begin{bmatrix}4\\1\\-2\end{bmatrix}.$$

Use it to write an equation for the plane.

:::{tip} Solution
:class: dropdown

Write $\vec n=\begin{bmatrix}a\\b\\c\end{bmatrix}$. Orthogonality to both spanning vectors gives

$$5a-7b+3c=0,\qquad 4a+b-2c=0.$$

The second equation gives $b=-4a+2c$. Substituting into the first gives $33a-11c=0$, so $c=3a$ and $b=2a$.

Choosing $a=1$ gives $\vec n=\begin{bmatrix}1\\2\\3\end{bmatrix}$. We can check that

$$\vec n\cdot\vec u=5-14+9=0,\qquad
\vec n\cdot\vec v=4+2-6=0.$$

The plane's equation is

$$x+2y+3z=0.$$
:::
::::

::::{tip} Activity 2
Find a nonzero normal to the plane spanned by

$$\vec u=\begin{bmatrix}6\\4\\-2\end{bmatrix},\qquad
\vec v=\begin{bmatrix}4\\12\\4\end{bmatrix}.$$

Use it to write an equation for the plane.

:::{tip} Solution
:class: dropdown

Write the unknown normal as

$$\vec n=\begin{bmatrix}a\\b\\c\end{bmatrix}.$$

Its entries must satisfy

$$6a+4b-2c=0,\qquad 4a+12b+4c=0.$$

The first equation gives

$$c=3a+2b.$$

Substitute this into the second equation:

$$4a+12b+4(3a+2b)=0\quad\Longrightarrow\quad16a+20b=0.$$

Choose $a=5$. Then

$$b=-4,\qquad c=7,\qquad \vec n=\begin{bmatrix}5\\-4\\7\end{bmatrix}.$$

Let's check both dot products:

$$\vec n\cdot\vec u=30-16-14=0,\qquad
\vec n\cdot\vec v=20-48+28=0.$$

So, the plane spanned by $\vec u = \begin{bmatrix}6\\4\\-2\end{bmatrix}$ and $\vec v = \begin{bmatrix}4\\12\\4\end{bmatrix}$ has the equation

$$5x-4y+7z=0.$$
:::
::::



---

## Lines as intersections of planes

We now have an understanding of how planes in $\mathbb{R}^3$ can be expressed:
- As the span of two linearly independent vectors in $\mathbb{R}^3$.
- As a linear equation, $$ax + by + cz = d$$ (so far, we've only seen the case where $d=0$.)

How do we describe **lines** using linear equations, when a single linear equation in terms of $x$, $y$, and $z$ describes a plane? That is what we will now explore. First, some terminology:

- A **linear system** is a collection of linear equations in the same unknowns.
- A **solution** gives values of the unknowns that satisfy every equation simultaneously. We typically express the solutions as vectors.
- The **solution set** is the set of all solutions – we often think of this as a set of vectors.
- A system is **homogeneous** if every right-hand side is zero. Otherwise, it is **nonhomogeneous**.

For a concrete example in $\mathbb R^2$, consider

$$\begin{cases}x+y=7,\\x-y=1.\end{cases}$$

Adding the equations gives $2x=8$, so $x=4$ and $y=3$. Thus, the solution set is

$$\left\{\begin{bmatrix}4\\3\end{bmatrix}\right\}.$$

Geometrically, the two lines intersect at the point $(4,3)$. The system is nonhomogeneous because its right-hand sides are not all zero.

Now let's return to $\mathbb R^3$. To describe a line, we need two linear equations and take their common solutions. Consider

$$\ell=\operatorname{span}\left(\begin{bmatrix}3\\4\\5\end{bmatrix}\right).$$

How do we find two equations? Pick two independent vectors $\vec n_1$ and $\vec n_2$ in $\ell^\perp$, the plane of vectors perpendicular to $\ell$.


In [7]:
fig=base3()
plane3(fig,(3,4,5),BLUE,0.22,bounds=(-5,5))
v=np.array([3,4,5])
line3(fig,-1.1*v,1.3*v,ORANGE,width=8)
vec3(fig,(1,-2,1),'<i>n</i>⃗<sub>1</sub>',PINK,offset=(0.2,-0.6,0.2))
vec3(fig,(2,1,-2),'<i>n</i>⃗<sub>2</sub>',PINK,offset=(0.4,0.2,-0.5))
fig.add_trace(go.Scatter3d(x=[3.9,-4],y=[5.2,4],z=[6.9,-0.8],mode='text',text=['ℓ','ℓ<sup>⊥</sup>'],textfont=dict(family='Palatino',size=25),hoverinfo='skip'))
fig.show()


```{figure} #plot-25-perpendicular-plane
:label: fig-plot-25-perpendicular-plane
:class: course-caption
:alt: An orange line perpendicular to a blue plane containing two pink normal vectors.

The two independent vectors $\vec n_1$ and $\vec n_2$ lie in $\ell^\perp$ (blue), perpendicular to the line $\ell$ (orange).
```


In our example, a vector $\begin{bmatrix}x\\y\\z\end{bmatrix}$ is in $\ell^\perp$ if

$$3x+4y+5z=0.$$

We can choose any values of $x$ and $y$ and solve for $z=-\frac{3x+4y}{5}$. For instance, choosing $x=1,y=-2$ gives $z=1$, while choosing $x=2,y=1$ gives $z=-2$. Thus we can use

$$\vec n_1=\begin{bmatrix}1\\-2\\1\end{bmatrix},\qquad
\vec n_2=\begin{bmatrix}2\\1\\-2\end{bmatrix}.$$

Both are perpendicular to $\begin{bmatrix}3\\4\\5\end{bmatrix}$, and they are not multiples of each other. The line is exactly the set of vectors orthogonal to both normals, so it is the solution set of

$$\begin{cases}x-2y+z=0,\\2x+y-2z=0.\end{cases}$$

What's happening geometrically? Each equation describes a plane:

$$P:\ x-2y+z=0,\qquad P':\ 2x+y-2z=0.$$

The line consists of the points on both planes. In other words,

$$\ell=P\cap P'.$$

The symbol $\cap$ means **intersection**.


In [8]:
def intersection_plot(normals,names):
    fig=base3()
    for normal,color in zip(normals,(BLUE,ORANGE)):
        plane3(fig,normal,color,0.28,bounds=(-5,7))
    v=np.array([3,4,5])
    line3(fig,-1.1*v,1.3*v,PINK,width=8)
    label_x = (5,-2) if normals[0] == (1,-2,1) else (1,-4)
    for normal,name,color,x,z in zip(normals,names,(BLUE,ORANGE),label_x,(-2,1)):
        y=-(normal[0]*x+normal[2]*z)/normal[1]
        fig.add_trace(go.Scatter3d(x=[x],y=[y],z=[z],mode='text',text=[name],textfont=dict(family='Palatino',size=26,color=color),hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(x=[4.1],y=[5.5],z=[6.9],mode='text',text=['ℓ'],textfont=dict(family='Palatino',size=26,color=PINK),hoverinfo='skip'))
    return fig

fig=intersection_plot([(1,-2,1),(2,1,-2)],['<i>P</i>',"<i>P</i>′"])
fig.show()


```{figure} #plot-25-intersection
:label: fig-plot-25-intersection
:class: course-caption
:alt: Two planes intersect along the pink line spanned by (3,4,5).

The blue plane $P$ and orange plane $P\prime$ intersect along the pink line $\ell$.
```


Our choices of normals were somewhat arbitrary. For example, we could instead use

$$\vec m_1=\vec n_1+\vec n_2=\begin{bmatrix}3\\-1\\-1\end{bmatrix},\qquad
\vec m_2=\vec n_1-\vec n_2=\begin{bmatrix}-1\\-3\\3\end{bmatrix}.$$

These are also perpendicular to $\ell$ and are not multiples of each other. They give another pair of planes,

$$Q:\ 3x-y-z=0,\qquad Q':\ -x-3y+3z=0,$$

whose intersection is the same line: $\ell=Q\cap Q'$. The two descriptions are shown side by side below.


In [9]:
from plotly.subplots import make_subplots
left=intersection_plot([(1,-2,1),(2,1,-2)],['<i>P</i>',"<i>P</i>′"])
right=intersection_plot([(3,-1,-1),(-1,-3,3)],['<i>Q</i>',"<i>Q</i>′"])
fig=make_subplots(rows=1,cols=2,specs=[[{'type':'scene'},{'type':'scene'}]],horizontal_spacing=0.03,subplot_titles=['ℓ = P ∩ P′','ℓ = Q ∩ Q′'])
for col,panel in enumerate((left,right),1):
    for trace in panel.data:
        fig.add_trace(trace,row=1,col=col)
fig.update_layout(template=left.layout.template,paper_bgcolor='white',font=dict(family='Palatino',size=16),height=480,margin=dict(l=0,r=0,t=45,b=0),showlegend=False)
scene=left.layout.scene.to_plotly_json()
lower=min(left.layout.scene.xaxis.range[0],right.layout.scene.xaxis.range[0])
upper=max(left.layout.scene.xaxis.range[1],right.layout.scene.xaxis.range[1])
for axis in ('xaxis','yaxis','zaxis'):
    scene[axis]['range']=[lower,upper]
fig.update_scenes(scene)
fig.show()


```{figure} #plot-25-two-intersections
:label: fig-plot-25-two-intersections
:class: course-caption
:alt: Side-by-side views of two different pairs of planes intersecting along the same pink line.

Different pairs of planes can have the same intersection: $\ell=P\cap P\prime=Q\cap Q\prime$. Drag either panel to rotate its view.
```


::::{tip} Activity 3
Find two independent vectors perpendicular to

$$\vec d=\begin{bmatrix}4\\-7\\6\end{bmatrix}.$$

Use them to describe $\operatorname{span}(\vec d)$ as the solution set of a linear system. Then find a second pair of independent perpendicular vectors to describe the same line with a different linear system.

:::{tip} Solution
:class: dropdown

The entries of a normal must satisfy

$$4a-7b+6c=0.$$

Setting $c=0,b=4$ gives $a=7$. Setting $b=0,c=-2$ gives $a=3$. Hence we can use

$$\vec n_1=\begin{bmatrix}7\\4\\0\end{bmatrix},\qquad
\vec n_2=\begin{bmatrix}3\\0\\-2\end{bmatrix}.$$

They are not scalar multiples, and

$$\vec n_1\cdot\vec d=28-28=0,\qquad
\vec n_2\cdot\vec d=12-12=0.$$

One system is

$$\begin{cases}7x+4y=0,\\3x-2z=0.\end{cases}$$

Setting $x=4t$ recovers the scalar-parametric form

$$x=4t,\qquad y=-7t,\qquad z=6t,\qquad t\in\mathbb R.$$

For a second choice, add and subtract our two normals:

$$\vec m_1=\vec n_1+\vec n_2=\begin{bmatrix}10\\4\\-2\end{bmatrix},\qquad
\vec m_2=\vec n_1-\vec n_2=\begin{bmatrix}4\\4\\2\end{bmatrix}.$$

Neither has a zero entry. They are not scalar multiples, and

$$\vec m_1\cdot\vec d=40-28-12=0,\qquad
\vec m_2\cdot\vec d=16-28+12=0.$$

A second system describing the same line is

$$\begin{cases}10x+4y-2z=0,\\4x+4y+2z=0.\end{cases}$$

Adding and subtracting these equations recovers twice each equation of the first system, so both systems have the same solution set.

:::
::::

We've now described lines and planes through the origin using linear equations (here in Chapter 2.5) as well as in parametric form (in [Chapter 2.4](./02-04.ipynb)). In [Chapter 2.6](02-06.ipynb), we'll translate these objects to study affine lines and planes, which do not need to pass through the origin.